In [ ]:
!pip install -q -U google-genai

import os
import json
from google import genai
from getpass import getpass

# ============================================================
# AI INTERVIEWER & PERFORMANCE EVALUATOR
# ============================================================

print("=" * 65)
print("        AI INTERVIEWER & PERFORMANCE EVALUATOR")
print("=" * 65)

# ============================================================
# GEMINI SETUP
# ============================================================

!pip install -q -U google-genai

from google import genai
from getpass import getpass

print("Enter your Gemini API key:")
API_KEY = getpass("API Key: ")

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"

print("Gemini connected successfully!")

# ------------------------------------------------------------
# STEP 2: CANDIDATE DETAILS
# ------------------------------------------------------------

print("\n--- Candidate Details ---")

name = input("Enter your name: ")
role = input("Enter the job role: ")
experience = input("Enter your experience level (Fresher/Intermediate/Experienced): ")
interview_type = input("Interview type (Technical/HR/Mixed): ")

print("\nCandidate registered successfully!")
print("Name:", name)
print("Role:", role)
print("Experience:", experience)
print("Interview Type:", interview_type)

# ============================================================
# STEP 3: GENERATE INTERVIEW QUESTIONS
# ============================================================

def generate_questions(name, role, experience, interview_type):

    prompt = f"""
You are an expert AI interviewer.

Create exactly 5 interview questions for the following candidate:

Candidate Name: {name}
Job Role: {role}
Experience Level: {experience}
Interview Type: {interview_type}

Requirements:
- Questions must match the candidate's job role.
- Questions should be suitable for the experience level.
- If Technical, ask technical questions.
- If HR, ask HR and behavioral questions.
- If Mixed, include both technical and HR questions.
- Questions should gradually increase in difficulty.
- Keep each question clear and concise.
- Do not provide answers.

Return only the 5 questions, one question per line.
"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )

        return response.text

    except Exception as e:
        print("\nERROR WHILE GENERATING QUESTIONS:")
        print(e)
        return None


# Generate questions
questions_text = generate_questions(
    name,
    role,
    experience,
    interview_type
)

if questions_text:

    print("\n" + "=" * 65)
    print("              AI GENERATED INTERVIEW QUESTIONS")
    print("=" * 65)

    print(questions_text)

else:
    print("\nCould not generate questions.")
    print("Check your Gemini API key and internet connection.")

# ============================================================
# STEP 4: PREPARE INTERVIEW QUESTIONS
# ============================================================

import re

# Make sure Step 3 generated something
if not questions_text:
    print("No questions were generated.")
    print("Please run Step 3 again.")
else:

    questions = []

    # Split the AI response into separate lines
    lines = questions_text.splitlines()

    for line in lines:

        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Remove numbering:
        # 1. Question
        # 2) Question
        # 3 - Question
        # 4: Question
        line = re.sub(
            r"^\s*(?:Question\s*)?\d+\s*[\.\)\-:]\s*",
            "",
            line,
            flags=re.IGNORECASE
        )

        # Remove markdown bullets
        line = re.sub(r"^\s*[-*•]\s*", "", line)

        # Only keep meaningful questions
        if len(line) > 10:
            questions.append(line)

    # Keep maximum 5 questions
    questions = questions[:5]

    # Check result
    if len(questions) == 0:

        print("Could not extract questions.")
        print("\nAI response was:")
        print(questions_text)

    else:

        print("\n" + "=" * 65)
        print("              INTERVIEW QUESTIONS READY")
        print("=" * 65)

        for i, question in enumerate(questions, start=1):
            print(f"\nQuestion {i}:")
            print(question)

        print("\nTotal questions:", len(questions))
        print("\nStep 4 completed successfully!")

# ------------------------------------------------------------
# STEP 5: INTERVIEW
# ------------------------------------------------------------

answers = []
evaluations = []

print("\n" + "=" * 65)
print("                    INTERVIEW STARTED")
print("=" * 65)

for i, question in enumerate(questions):

    print(f"\nQuestion {i + 1}:")
    print(question)

    answer = input("\nYour Answer: ")

    if answer.strip() == "":
        answer = "No answer provided."

    answers.append(answer)

    # --------------------------------------------------------
    # STEP 6: AI EVALUATION
    # --------------------------------------------------------

    evaluation_prompt = f"""
You are an AI interview evaluator.

Evaluate the following candidate answer.

Candidate:
{name}

Job Role:
{role}

Question:
{question}

Candidate Answer:
{answer}

Give scores from 0 to 10 for:

1. Technical Knowledge
2. Accuracy
3. Relevance
4. Communication
5. Confidence
6. Problem Solving

Then calculate an overall score out of 10.

Also provide:
- What was good
- What can be improved
- A short suggestion

Return the result in this exact format:

Technical Knowledge: X/10
Accuracy: X/10
Relevance: X/10
Communication: X/10
Confidence: X/10
Problem Solving: X/10
Overall Score: X/10

Good Points:
...

Improvements:
...

Suggestion:
...
"""

    evaluation = client.models.generate_content(
        model=MODEL,
        contents=evaluation_prompt
    )

    evaluations.append(evaluation.text)

    print("\n--- AI Evaluation ---")
    print(evaluation.text)

# ------------------------------------------------------------
# STEP 7: FINAL PERFORMANCE REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("                 FINAL PERFORMANCE REPORT")
print("=" * 65)

all_data = ""

for i in range(len(questions)):
    all_data += f"""
Question {i + 1}: {questions[i]}

Candidate Answer:
{answers[i]}

Evaluation:
{evaluations[i]}

-----------------------------------------
"""

final_prompt = f"""
You are a professional interview performance evaluator.

Prepare a final interview report for:

Candidate Name: {name}
Job Role: {role}
Experience: {experience}
Interview Type: {interview_type}

Here are the interview results:

{all_data}

Create a clear final report containing:

1. Candidate Name
2. Job Role
3. Overall Score out of 100
4. Technical Performance
5. Communication Performance
6. Problem-Solving Performance
7. Confidence
8. Strong Areas
9. Weak Areas
10. Recommended Improvements
11. Final Interview Result

For the final result use one of:
- SELECTED
- NEEDS IMPROVEMENT
- NOT SELECTED

Use the following approximate rule:
80-100 = SELECTED
60-79 = NEEDS IMPROVEMENT
Below 60 = NOT SELECTED

Keep the report professional and easy to understand.
"""

final_report = client.models.generate_content(
    model=MODEL,
    contents=final_prompt
)

print(final_report.text)

# ------------------------------------------------------------
# STEP 8: SAVE REPORT
# ------------------------------------------------------------

report_filename = "AI_Interview_Performance_Report.txt"

with open(report_filename, "w", encoding="utf-8") as file:

    file.write("=" * 65 + "\n")
    file.write("       AI INTERVIEWER & PERFORMANCE EVALUATOR\n")
    file.write("=" * 65 + "\n\n")

    file.write(f"Candidate Name: {name}\n")
    file.write(f"Job Role: {role}\n")
    file.write(f"Experience: {experience}\n")
    file.write(f"Interview Type: {interview_type}\n\n")

    file.write("=" * 65 + "\n")
    file.write("INTERVIEW QUESTIONS AND EVALUATIONS\n")
    file.write("=" * 65 + "\n\n")

    for i in range(len(questions)):

        file.write(f"Question {i + 1}: {questions[i]}\n\n")
        file.write(f"Candidate Answer:\n{answers[i]}\n\n")
        file.write(f"AI Evaluation:\n{evaluations[i]}\n\n")
        file.write("-" * 65 + "\n\n")

    file.write("=" * 65 + "\n")
    file.write("FINAL PERFORMANCE REPORT\n")
    file.write("=" * 65 + "\n\n")

    file.write(final_report.text)

print("\nReport saved successfully!")

# ------------------------------------------------------------
# STEP 9: DOWNLOAD REPORT IN GOOGLE COLAB
# ------------------------------------------------------------

try:
    from google.colab import files
    files.download(report_filename)
except:
    print("If you are not using Google Colab, the report is saved as:")
    print(report_filename)

print("\n" + "=" * 65)
print("              INTERVIEW COMPLETED")
print("=" * 65)

        AI INTERVIEWER & PERFORMANCE EVALUATOR
Enter your Gemini API key:
Gemini connected successfully!

--- Candidate Details ---

Candidate registered successfully!
Name: Monikyuy
Role: developer
Experience: fresher
Interview Type: technical

              AI GENERATED INTERVIEW QUESTIONS
What is the difference between a programming language and a markup language?
Can you explain what a variable is and give an example of how it is used in code?
How would you find and fix a bug in a simple program that is not producing the expected output?
What is the purpose of using functions or methods in software development?
Could you write a simple algorithm or pseudo-code to determine if a given number is even or odd?

              INTERVIEW QUESTIONS READY

Question 1:
What is the difference between a programming language and a markup language?

Question 2:
Can you explain what a variable is and give an example of how it is used in code?

Question 3:
How would you find and fix a bug in a sim